### This file is used to insert problems into the DB table Problems

In [1]:
import json
import psycopg
from psycopg.types.json import Json

##### Make DB connection


In [2]:
conn = psycopg.connect(
    dbname = "leetlog",
    user = "postgres",
    password="root",
    host="localhost"
)

#### Making a cursor object 

In [3]:
cur = conn.cursor()

### How to insert Data into the  Problems Tables from json files?
####  -> WE user a strategy as  Upsert , If a problems already exists in the DB we update the problems , if it does not , we insert it new .
UPSERT in PostgreSQL is a database operation that combines the functionality of INSERT and UPDATE.
It allows us to either insert a new row into a table or update an existing row if it already exists, all in a single command.
This is particularly useful for maintaining data integrity and avoiding duplicate entries.

In [4]:
with open("../dsa_data.json","r",encoding="utf-8") as file:
    data = json.load(file)
with open("../dsa_descriptions.json","r") as file:
    description = json.load(file)
    
    
if data and description :
    print("data loaded 👍")

data loaded 👍


In [5]:
for topic,problem in data.items():
    print(problem.keys())

dict_keys(['Aggressive-cows.cpp', 'Allocate-Minimum-Pages.cpp', 'Count of Smaller Numbers After Self.cpp', 'Find Right Interval.cpp', 'Find Smallest Letter Greater Than Target.cpp', 'Find a Peak Element II.cpp', 'Find the Duplicate Number.cpp', 'Find-Minimum-in-Rotated-Sorted-Array-II.cpp', 'First and Last Occurrences.cpp', 'First-Bad-Version.cpp', 'H-Index II.cpp', 'Heaters.cpp', 'Intersection-of-Two-Arrays.cpp', 'K-diff Pairs in an Array.cpp', 'K-th element of two Arrays.cpp', 'Kth Smallest Element in a Sorted Matrix.cpp', 'Kth Smallest Product of Two Sorted Arrays.cpp', 'Kth-Missing-Positive-Number.cpp', 'Median in a row-wise sorted Matrix.cpp', 'Median-of-Two-Sorted-Arrays.cpp', 'Minimize Max Distance to Gas Station.cpp', 'Minimum Size Subarray Sum.cpp', 'Nim game.cpp', 'Number of occurrence.cpp', 'Row With Maximum Ones.cpp', 'Search a 2D Matrix.cpp', 'Search in Rotated Sorted Array.cpp', 'Search-Insert-Position.cpp', 'Split-Array-Largest-Sum.cpp', 'Sqrt(x).cpp'])
dict_keys(['Add N

In [6]:
YELLOW = '\033[93m'
RESET = '\033[0m'

for topic,problems in data.items():
    for problem_name ,code in problems.items():
        problem_data  =  description.get(problem_name)
        print(f"{YELLOW}Topic:{RESET} {topic}\n")
        print(f"{YELLOW}Problem Name:{RESET} {problem_name}\n")    
        print(f"{YELLOW}Title:{RESET} {problem_data.get('title')}")       
        print(f"{YELLOW}Source:{RESET} {problem_data.get('source')}")       
        print(f"{YELLOW}Link:{RESET} {problem_data.get('link')}")       
        print(f"{YELLOW}Description:{RESET} {problem_data.get('description')}")       
        print(f"{YELLOW}Constraints:{RESET} {problem_data.get('constraints')}")       
        print(f"{YELLOW}Difficulty:{RESET} {problem_data.get('difficulty')}")       
        print(f"{YELLOW}Examples:{RESET} {problem_data.get('examples')}\n")       
        print(f"{YELLOW}Code:{RESET}\n{code}\n")
        print("-" * 50)

Topic: Binary Search

Problem Name: Aggressive-cows.cpp

Title: Aggressive Cows
Source: GFG
Link: https://www.geeksforgeeks.org/problems/aggressive-cows/0
Description: You are given an array stalls[] of unique integers denoting the positions of stalls. You are also given an integer k which denotes the number of aggressive cows. Assign stalls to k cows such that the minimum distance between any two of them is the maximum possible.
Constraints: ['2 â‰¤ arr.size() â‰¤ 10^6', '0 â‰¤ arr[i] â‰¤ 10^8', '2 â‰¤ k â‰¤ arr.size()']
Difficulty: Medium
Examples: [{'input': 'stalls[] = [1, 2, 4, 8, 9], k = 3', 'output': '3', 'explanation': 'Place cows at stalls[0]=1, stalls[2]=4, stalls[3]=8. Minimum distance = 3, which is the largest possible.'}, {'input': 'stalls[] = [10, 1, 2, 7, 5], k = 3', 'output': '4', 'explanation': 'Place cows at positions 1, 5, 10. Minimum distance = 4, which is the largest possible.'}]

Code:
 //Brute force TC: O(n×maxDist) SC:O(1) 
class Solution {
  public:
  bool chec

In [7]:
upsert_query = """INSERT INTO Problems (
                                                problem_key,
                                                title,
                                                topic,
                                                difficulty,
                                                source,
                                                link,
                                                description,
                                                constraints,
                                                examples,
                                                editorial_code,
                                                time_limit_ms,
                                                memory_limit_mb
                                                )
                            values (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                            ON CONFLICT(problem_key)
                            DO UPDATE SET
                            title = EXCLUDED.title,
                            topic = EXCLUDED.topic,
                            difficulty = EXCLUDED.difficulty,
                            source = EXCLUDED.source,
                            link = EXCLUDED.link,
                            description = EXCLUDED.description,
                            constraints=  EXCLUDED.constraints,
                            examples = EXCLUDED.examples,
                            editorial_code = EXCLUDED.editorial_code,
                            time_limit_ms = EXCLUDED.time_limit_ms,
                            memory_limit_mb = EXCLUDED.memory_limit_mb
                            """

time_limit_ms = 2000
memory_limit_mb = 256
cnt = 0
for topic,problems in data.items():
    for problem_name ,code in problems.items():
        problem_data = description.get(problem_name)
        problem_key = problem_name.removesuffix(".cpp")
        title = problem_data.get("title",problem_key).removesuffix(".cpp")
        source = problem_data.get("source","Not Available")
        link = problem_data.get("link","https://leetcode.com/problemset/")
        desc = problem_data.get("description","Not Available")
        constraints = problem_data.get("constraints",["Constraints not available."])
        difficulty = problem_data.get("difficulty","Medium")
        
        examples = problem_data.get("examples",[{
            "input":"Not Available",
            "output":"Not Available",
            "explanation":"Explanation Not Available"
        }])
        
        values = (problem_key,title,topic,difficulty,source,link,desc,Json(constraints),Json(examples),code,time_limit_ms,memory_limit_mb)
        try:
            cur.execute(upsert_query,values)
            print(f"✅: {problem_key}")
            cnt=cnt+1
        except Exception as e:
            print(f"❌ {problem_key}: {e}")
            conn.rollback()
            break


conn.commit()
print("--------------------------------")
print("Import completed successfully!")
print(f"Total Inserts: {cnt}")
print("--------------------------------")



✅: Aggressive-cows
✅: Allocate-Minimum-Pages
✅: Count of Smaller Numbers After Self
✅: Find Right Interval
✅: Find Smallest Letter Greater Than Target
✅: Find a Peak Element II
✅: Find the Duplicate Number
✅: Find-Minimum-in-Rotated-Sorted-Array-II
✅: First and Last Occurrences
✅: First-Bad-Version
✅: H-Index II
✅: Heaters
✅: Intersection-of-Two-Arrays
✅: K-diff Pairs in an Array
✅: K-th element of two Arrays
✅: Kth Smallest Element in a Sorted Matrix
✅: Kth Smallest Product of Two Sorted Arrays
✅: Kth-Missing-Positive-Number
✅: Median in a row-wise sorted Matrix
✅: Median-of-Two-Sorted-Arrays
✅: Minimize Max Distance to Gas Station
✅: Minimum Size Subarray Sum
✅: Nim game
✅: Number of occurrence
✅: Row With Maximum Ones
✅: Search a 2D Matrix
✅: Search in Rotated Sorted Array
✅: Search-Insert-Position
✅: Split-Array-Largest-Sum
✅: Sqrt(x)
✅: Add Number Linked Lists
✅: Add one to a linked list number
✅: Convert Binary Number in a Linked List to Integer
✅: Copy List with Random Pointer
✅

In [8]:
cur.execute("SELECT COUNT(*) FROM Problems")

<psycopg.Cursor [TUPLES_OK] [INTRANS] (host=localhost user=postgres database=leetlog) at 0x18c10f4f410>

In [9]:
cur.fetchone()

(174,)